# 知识蒸馏教程 (Knowledge Distillation Tutorial)

> **前置知识**: PyTorch 基础、深度学习模型训练流程
>
> **学习目标**: 掌握知识蒸馏的原理、损失函数设计和实现方法

---

## 为什么需要知识蒸馏？

```
┌─────────────────────────────────────────────────────────────┐
│                   知识蒸馏的核心价值                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  问题: 大模型精度高但部署困难                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  教师模型 (Teacher)                                 │   │
│  │  - 参数量: 数十亿                                   │   │
│  │  - 精度: 高                                         │   │
│  │  - 推理速度: 慢                                     │   │
│  │  - 部署成本: 高                                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                         ↓ 知识蒸馏                          │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  学生模型 (Student)                                 │   │
│  │  - 参数量: 数百万 (压缩 10-100x)                    │   │
│  │  - 精度: 接近教师 (损失 <2%)                        │   │
│  │  - 推理速度: 快                                     │   │
│  │  - 部署成本: 低                                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  核心思想: 让小模型学习大模型的"知识"，而非原始标签         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **软标签与温度参数** - 理解"暗知识"的概念
2. **响应蒸馏** - 匹配输出分布 (最基础)
3. **特征蒸馏** - 匹配中间层特征
4. **关系蒸馏** - 保持样本间关系
5. **完整蒸馏训练** - 端到端实现

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"PyTorch 版本: {torch.__version__}")

## 1. 软标签与温度参数

**核心概念**: 软标签包含类别之间的相似性信息，这是"暗知识"的来源

```
┌─────────────────────────────────────────────────────────────┐
│                 硬标签 vs 软标签                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  硬标签 (Hard Labels) - 传统监督学习                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  y = [0, 0, 1, 0, 0]  (one-hot 编码)               │   │
│  │  只告诉模型"这是猫"，没有其他信息                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  软标签 (Soft Labels) - 知识蒸馏                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  p = [0.05, 0.15, 0.70, 0.08, 0.02]                │   │
│  │  告诉模型"这是猫，但有点像狗，不太像鸟"            │   │
│  │  包含丰富的类间关系信息 (暗知识)                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  温度参数 T 的作用:                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  p_i = exp(z_i / T) / Σ exp(z_j / T)               │   │
│  │                                                     │   │
│  │  T = 1: 标准 softmax，分布较尖锐                   │   │
│  │  T > 1: 分布更平滑，暴露更多类间关系               │   │
│  │  T < 1: 分布更尖锐，接近 one-hot                   │   │
│  │                                                     │   │
│  │  典型值: T = 4 ~ 20                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 温度参数对软标签的影响
# ============================================================
print("=" * 60)
print("温度参数对软标签的影响")
print("=" * 60)

# 模拟教师模型输出 (logits)
# 假设这是一个5分类问题，教师模型认为类别0最可能
teacher_logits = torch.tensor([[2.0, 1.0, 0.5, -1.0, -2.0]])

# 不同温度下的软标签
temperatures = [1.0, 2.0, 4.0, 8.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

for ax, T in zip(axes, temperatures):
    # 使用温度 T 计算软标签
    # p_i = exp(z_i / T) / Σ exp(z_j / T)
    soft_labels = F.softmax(teacher_logits / T, dim=-1).squeeze().numpy()
    
    ax.bar(range(5), soft_labels, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_title(f'温度 T = {T}', fontsize=12)
    ax.set_xlabel('类别')
    ax.set_ylabel('概率')
    ax.set_ylim(0, 1)
    
    # 标注最大概率
    ax.text(0, soft_labels[0] + 0.02, f'{soft_labels[0]:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n观察:")
print("  T=1: 分布尖锐，类别0概率最高 (接近 one-hot)")
print("  T=8: 分布平滑，暴露更多类间关系信息")
print("  → 高温度让学生学到更多'暗知识'")

In [ ]:
# ============================================================
# 温度参数的数学解释
# ============================================================
print("=" * 60)
print("温度参数的数学解释")
print("=" * 60)
print(f"\n{'温度':<10} {'最大概率':<15} {'熵':<15} {'分布特点':<20}")
print("-" * 60)

for T in [0.5, 1.0, 2.0, 4.0, 8.0]:
    probs = F.softmax(teacher_logits / T, dim=-1).squeeze()
    max_prob = probs.max().item()
    # 熵: H = -Σ p_i * log(p_i)，衡量分布的不确定性
    entropy = -(probs * probs.log()).sum().item()
    
    if T < 1:
        desc = "更尖锐"
    elif T == 1:
        desc = "标准 softmax"
    else:
        desc = "更平滑"
    
    print(f"{T:<10} {max_prob:<15.4f} {entropy:<15.4f} {desc:<20}")

print("\n解释:")
print("  - 熵越高，分布越均匀，包含更多'暗知识'")
print("  - 典型蒸馏温度: T=4~20")
print("  - 温度过高会丢失主要类别信息，过低则暗知识不足")

## 2. 定义教师和学生模型

**核心概念**: 教师模型大而准确，学生模型小而高效

```
┌─────────────────────────────────────────────────────────────┐
│                   模型架构对比                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  教师模型 (Teacher)                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Input(784) → FC(512) → FC(512) → FC(256) → Out(10) │   │
│  │  参数量: ~660K                                      │   │
│  │  特点: 深层网络，表达能力强                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  学生模型 (Student)                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Input(784) → FC(128) → Out(10)                     │   │
│  │  参数量: ~100K                                      │   │
│  │  特点: 浅层网络，推理速度快                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  压缩比: ~6.6x                                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义教师和学生模型
# ============================================================

class TeacherModel(nn.Module):
    """
    教师模型 (较大)
    
    结构: 4层全连接网络
    特点: 深层网络，表达能力强，但推理慢
    """
    def __init__(self, input_dim=784, hidden_dim=512, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)      # 784 → 512
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)     # 512 → 512
        self.fc3 = nn.Linear(hidden_dim, hidden_dim // 2) # 512 → 256
        self.fc4 = nn.Linear(hidden_dim // 2, num_classes) # 256 → 10
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        x = self.relu(self.fc3(x))
        return self.fc4(x)


class StudentModel(nn.Module):
    """
    学生模型 (较小)
    
    结构: 2层全连接网络
    特点: 浅层网络，推理速度快，但表达能力有限
    """
    def __init__(self, input_dim=784, hidden_dim=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)      # 784 → 128
        self.fc2 = nn.Linear(hidden_dim, num_classes)    # 128 → 10
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)


# ============================================================
# 创建模型并比较参数量
# ============================================================
teacher = TeacherModel()
student = StudentModel()

# 统计参数
teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print("=" * 60)
print("模型参数对比")
print("=" * 60)
print(f"\n教师模型:")
print(f"  结构: 784 → 512 → 512 → 256 → 10")
print(f"  参数量: {teacher_params:,}")

print(f"\n学生模型:")
print(f"  结构: 784 → 128 → 10")
print(f"  参数量: {student_params:,}")

print(f"\n压缩比: {teacher_params / student_params:.1f}x")
print(f"学生模型只有教师模型 {student_params / teacher_params * 100:.1f}% 的参数")

## 3. 响应蒸馏 (Response-based Distillation)

**核心概念**: 最基本的蒸馏方法，让学生模型学习教师模型的输出分布

```
┌─────────────────────────────────────────────────────────────┐
│                   响应蒸馏损失函数                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  总损失 = α × L_soft + (1-α) × L_hard                      │
│                                                             │
│  L_soft (软标签损失):                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  KL散度: KL(P_student || P_teacher)                 │   │
│  │  P = softmax(logits / T)                            │   │
│  │  需要乘以 T² 补偿梯度缩放                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  L_hard (硬标签损失):                                       │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  交叉熵: CrossEntropy(student_logits, labels)       │   │
│  │  确保学生仍然学习正确的分类                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  参数说明:                                                  │
│  - T: 温度参数，典型值 4~20                                │
│  - α: 软标签权重，典型值 0.5~0.9                           │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 蒸馏损失函数实现
# ============================================================

def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    """
    知识蒸馏损失函数
    
    参数:
        student_logits: 学生模型输出 logits [batch_size, num_classes]
        teacher_logits: 教师模型输出 logits [batch_size, num_classes]
        labels: 真实标签 [batch_size]
        temperature: 温度参数，控制软标签平滑度
        alpha: 软标签损失权重 (0-1)
        
    返回:
        total_loss: 总损失
        loss_dict: 损失分解字典
        
    损失公式:
    ┌─────────────────────────────────────────────────────────┐
    │  L_total = α × L_soft + (1-α) × L_hard                 │
    │                                                         │
    │  L_soft = KL(P_student || P_teacher) × T²              │
    │  L_hard = CrossEntropy(student_logits, labels)         │
    └─────────────────────────────────────────────────────────┘
    """
    # ============================================================
    # 软标签损失 (KL 散度)
    # ============================================================
    # 使用温度 T 软化 logits
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    
    # KL 散度: KL(P || Q) = Σ P * log(P/Q)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean')
    
    # 重要: 乘以 T² 补偿梯度缩放
    # 原因: softmax 在高温下梯度会缩小 1/T，需要补偿
    soft_loss = soft_loss * (temperature ** 2)
    
    # ============================================================
    # 硬标签损失 (交叉熵)
    # ============================================================
    hard_loss = F.cross_entropy(student_logits, labels)
    
    # ============================================================
    # 组合损失
    # ============================================================
    total_loss = alpha * soft_loss + (1 - alpha) * hard_loss
    
    loss_dict = {
        'soft_loss': soft_loss.item(),
        'hard_loss': hard_loss.item(),
        'total_loss': total_loss.item()
    }
    
    return total_loss, loss_dict


# ============================================================
# 测试蒸馏损失
# ============================================================
print("=" * 60)
print("蒸馏损失函数测试")
print("=" * 60)

# 模拟数据
x = torch.randn(32, 784)
labels = torch.randint(0, 10, (32,))

# 前向传播
teacher.eval()
with torch.no_grad():
    teacher_logits = teacher(x)

student.train()
student_logits = student(x)

# 计算损失
total_loss, loss_dict = distillation_loss(
    student_logits, teacher_logits, labels,
    temperature=4.0, alpha=0.7
)

print(f"\n蒸馏损失分解:")
print(f"  软标签损失 (KL): {loss_dict['soft_loss']:.4f}")
print(f"  硬标签损失 (CE): {loss_dict['hard_loss']:.4f}")
print(f"  总损失: {loss_dict['total_loss']:.4f}")
print(f"\n损失权重: α={0.7} (软标签), 1-α={0.3} (硬标签)")

In [ ]:
# ============================================================
# 训练教师模型 (模拟预训练)
# ============================================================
print("=" * 60)
print("训练教师模型")
print("=" * 60)

# 创建模拟训练数据
# 实际应用中应使用真实数据集 (如 MNIST, CIFAR-10)
train_data = [(torch.randn(64, 784), torch.randint(0, 10, (64,))) for _ in range(100)]

# 训练教师模型
teacher = TeacherModel()
teacher_optimizer = torch.optim.Adam(teacher.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("\n训练教师模型...")
teacher.train()
for epoch in range(5):
    total_loss = 0
    for x_batch, y_batch in train_data:
        teacher_optimizer.zero_grad()
        output = teacher(x_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        teacher_optimizer.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_data)
    print(f"  Epoch {epoch+1}/5: Loss = {avg_loss:.4f}")

print("\n教师模型训练完成!")
print("注意: 实际应用中，教师模型应该是一个预训练好的高精度模型")

In [ ]:
# ============================================================
# 完整的知识蒸馏训练
# ============================================================
print("=" * 60)
print("知识蒸馏训练")
print("=" * 60)

# 创建新的学生模型
student = StudentModel()
student_optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

# 蒸馏参数
temperature = 4.0
alpha = 0.7
num_epochs = 10

# 记录训练历史
history = []

print(f"\n蒸馏参数:")
print(f"  温度 T = {temperature}")
print(f"  软标签权重 α = {alpha}")
print(f"  训练轮数 = {num_epochs}")

print(f"\n开始蒸馏训练...")
teacher.eval()  # 教师模型固定为评估模式

for epoch in range(num_epochs):
    student.train()
    epoch_soft_loss = 0
    epoch_hard_loss = 0
    epoch_total_loss = 0
    
    for x_batch, y_batch in train_data:
        # 获取教师模型输出 (不需要梯度)
        with torch.no_grad():
            teacher_logits = teacher(x_batch)
        
        # 学生模型前向传播
        student_logits = student(x_batch)
        
        # 计算蒸馏损失
        loss, loss_dict = distillation_loss(
            student_logits, teacher_logits, y_batch,
            temperature=temperature, alpha=alpha
        )
        
        # 反向传播
        student_optimizer.zero_grad()
        loss.backward()
        student_optimizer.step()
        
        epoch_soft_loss += loss_dict['soft_loss']
        epoch_hard_loss += loss_dict['hard_loss']
        epoch_total_loss += loss_dict['total_loss']
    
    # 记录历史
    n = len(train_data)
    history.append({
        'soft': epoch_soft_loss / n,
        'hard': epoch_hard_loss / n,
        'total': epoch_total_loss / n
    })
    
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/{num_epochs}: "
              f"Total={history[-1]['total']:.4f}, "
              f"Soft={history[-1]['soft']:.4f}, "
              f"Hard={history[-1]['hard']:.4f}")

print("\n蒸馏训练完成!")

In [ ]:
# ============================================================
# 可视化训练历史
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(history) + 1)

# 图1: 总损失曲线
axes[0].plot(epochs, [h['total'] for h in history], 'b-', linewidth=2, marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('损失')
axes[0].set_title('蒸馏训练: 总损失曲线', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 图2: 损失分解
axes[1].plot(epochs, [h['soft'] for h in history], 'g-', linewidth=2, marker='s', label='软标签损失 (KL)')
axes[1].plot(epochs, [h['hard'] for h in history], 'r-', linewidth=2, marker='^', label='硬标签损失 (CE)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('损失')
axes[1].set_title('蒸馏训练: 损失分解', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n观察:")
print("  - 软标签损失: 学生学习教师的输出分布")
print("  - 硬标签损失: 学生学习正确的分类")
print("  - 两者共同作用，使学生既学到知识又保持准确性")

## 4. 特征蒸馏 (Feature-based Distillation)

**核心概念**: 除了输出层，还可以匹配中间层特征，传递更多知识

```
┌─────────────────────────────────────────────────────────────┐
│                   特征蒸馏原理                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  教师模型                        学生模型                   │
│  ┌─────────┐                    ┌─────────┐                │
│  │ Layer 1 │ ──────────────────→│ Layer 1 │  特征匹配      │
│  ├─────────┤                    ├─────────┤                │
│  │ Layer 2 │ ──────────────────→│         │  (可选)        │
│  ├─────────┤                    │         │                │
│  │ Layer 3 │                    │         │                │
│  ├─────────┤                    ├─────────┤                │
│  │ Output  │ ──────────────────→│ Output  │  响应匹配      │
│  └─────────┘                    └─────────┘                │
│                                                             │
│  损失函数:                                                  │
│  L_feature = MSE(student_features, teacher_features)       │
│                                                             │
│  注意: 教师和学生的特征维度可能不同，需要投影层对齐         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 特征蒸馏实现
# ============================================================

class FeatureDistillation(nn.Module):
    """
    特征蒸馏模块
    
    匹配教师和学生的中间层特征
    如果维度不同，使用投影层对齐
    """
    def __init__(self, student_dim, teacher_dim):
        super().__init__()
        # 如果维度不同，添加投影层
        if student_dim != teacher_dim:
            self.projector = nn.Linear(student_dim, teacher_dim)
        else:
            self.projector = None
    
    def forward(self, student_features, teacher_features):
        """
        计算特征蒸馏损失
        
        参数:
            student_features: 学生特征 [batch_size, student_dim]
            teacher_features: 教师特征 [batch_size, teacher_dim]
            
        返回:
            MSE 损失
        """
        # 投影学生特征到教师维度
        if self.projector is not None:
            student_features = self.projector(student_features)
        
        # 计算 MSE 损失
        return F.mse_loss(student_features, teacher_features)


# ============================================================
# 测试特征蒸馏
# ============================================================
print("=" * 60)
print("特征蒸馏演示")
print("=" * 60)

# 创建特征蒸馏模块
feature_distiller = FeatureDistillation(
    student_dim=128,   # 学生特征维度
    teacher_dim=512    # 教师特征维度
)

# 模拟特征
student_features = torch.randn(32, 128)
teacher_features = torch.randn(32, 512)

# 计算特征蒸馏损失
feature_loss = feature_distiller(student_features, teacher_features)

print(f"\n学生特征维度: {student_features.shape}")
print(f"教师特征维度: {teacher_features.shape}")
print(f"特征蒸馏损失: {feature_loss.item():.4f}")
print(f"\n投影层: 128 → 512 (对齐维度)")

In [ ]:
# ============================================================
# 注意力迁移 (Attention Transfer) - 用于 CNN
# ============================================================

class AttentionTransfer(nn.Module):
    """
    注意力迁移
    
    用于 CNN 模型，匹配特征图的空间注意力
    注意力图 = 特征图沿通道维度的 p 范数
    """
    def __init__(self, p=2):
        super().__init__()
        self.p = p  # 范数类型
    
    def attention_map(self, feature_map):
        """
        计算注意力图
        
        参数:
            feature_map: 特征图 [batch, channels, H, W]
            
        返回:
            attention: 注意力图 [batch, H, W]
        """
        # 沿通道维度计算 p 范数
        attention = torch.norm(feature_map, p=self.p, dim=1)
        # 归一化
        attention = attention / (attention.sum(dim=(1, 2), keepdim=True) + 1e-8)
        return attention
    
    def forward(self, student_fmap, teacher_fmap):
        """
        计算注意力迁移损失
        """
        student_attention = self.attention_map(student_fmap)
        teacher_attention = self.attention_map(teacher_fmap)
        return F.mse_loss(student_attention, teacher_attention)


# ============================================================
# 测试注意力迁移
# ============================================================
print("=" * 60)
print("注意力迁移演示 (用于 CNN)")
print("=" * 60)

attention_transfer = AttentionTransfer(p=2)

# 模拟 CNN 特征图
student_fmap = torch.randn(8, 32, 7, 7)   # [batch, channels, H, W]
teacher_fmap = torch.randn(8, 64, 7, 7)

# 计算注意力图
student_attention = attention_transfer.attention_map(student_fmap)
teacher_attention = attention_transfer.attention_map(teacher_fmap)

print(f"\n学生特征图形状: {student_fmap.shape}")
print(f"教师特征图形状: {teacher_fmap.shape}")
print(f"学生注意力图形状: {student_attention.shape}")
print(f"教师注意力图形状: {teacher_attention.shape}")

# 注意力迁移损失
at_loss = attention_transfer(student_fmap, teacher_fmap)
print(f"\n注意力迁移损失: {at_loss.item():.4f}")
print("\n注意: 注意力图只关注空间位置，不关注通道数，因此可以跨不同通道数匹配")

## 5. 关系蒸馏 (Relation-based Distillation)

**核心概念**: 保持样本之间的关系结构，而非匹配具体特征值

```
┌─────────────────────────────────────────────────────────────┐
│                   关系蒸馏原理                               │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  核心思想: 样本间的关系比具体特征值更重要                   │
│                                                             │
│  样本 A, B, C 在教师模型中的关系:                           │
│  ┌─────────────────────────────────────────────────────┐   │
│  │       A                                             │   │
│  │      / \                                            │   │
│  │     /   \  相似度矩阵                               │   │
│  │    B─────C                                          │   │
│  │                                                     │   │
│  │  sim(A,B) = 0.8                                    │   │
│  │  sim(A,C) = 0.3                                    │   │
│  │  sim(B,C) = 0.5                                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  目标: 让学生模型保持相同的关系结构                         │
│                                                             │
│  损失函数:                                                  │
│  L_relation = MSE(student_similarity, teacher_similarity)  │
│                                                             │
│  优点: 不依赖具体特征值，更鲁棒                             │
│  缺点: 计算复杂度 O(n²)                                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 关系蒸馏实现
# ============================================================

class RelationDistillation(nn.Module):
    """
    关系蒸馏
    
    保持样本之间的相似度关系
    """
    def __init__(self, distance_type="cosine"):
        super().__init__()
        self.distance_type = distance_type
    
    def compute_similarity_matrix(self, features):
        """
        计算样本间的相似度矩阵
        
        参数:
            features: 特征 [batch_size, feature_dim]
            
        返回:
            similarity: 相似度矩阵 [batch_size, batch_size]
        """
        if self.distance_type == "cosine":
            # 余弦相似度
            features_norm = F.normalize(features, p=2, dim=1)
            similarity = torch.mm(features_norm, features_norm.t())
        else:
            # 欧氏距离 (转换为相似度)
            dist = torch.cdist(features, features, p=2)
            similarity = 1 / (1 + dist)
        return similarity
    
    def forward(self, student_features, teacher_features):
        """
        计算关系蒸馏损失
        """
        student_sim = self.compute_similarity_matrix(student_features)
        teacher_sim = self.compute_similarity_matrix(teacher_features)
        return F.mse_loss(student_sim, teacher_sim)


# ============================================================
# 测试关系蒸馏
# ============================================================
print("=" * 60)
print("关系蒸馏演示")
print("=" * 60)

relation_distiller = RelationDistillation(distance_type="cosine")

# 模拟特征 (注意: 维度可以不同)
student_features = torch.randn(16, 128)
teacher_features = torch.randn(16, 512)

# 计算相似度矩阵
student_sim = relation_distiller.compute_similarity_matrix(student_features)
teacher_sim = relation_distiller.compute_similarity_matrix(teacher_features)

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(student_sim.detach().numpy(), cmap='coolwarm', vmin=-1, vmax=1)
axes[0].set_title('学生模型相似度矩阵', fontsize=12)
axes[0].set_xlabel('样本索引')
axes[0].set_ylabel('样本索引')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(teacher_sim.detach().numpy(), cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_title('教师模型相似度矩阵', fontsize=12)
axes[1].set_xlabel('样本索引')
axes[1].set_ylabel('样本索引')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

# 关系蒸馏损失
relation_loss = relation_distiller(student_features, teacher_features)
print(f"\n关系蒸馏损失: {relation_loss.item():.4f}")
print("\n注意: 关系蒸馏不要求特征维度相同，只要求保持样本间关系")

## 6. 蒸馏 vs 直接训练对比

**核心问题**: 蒸馏真的比直接训练学生模型更好吗？

```
┌─────────────────────────────────────────────────────────────┐
│                   蒸馏 vs 直接训练                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  直接训练:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  学生模型 ← 硬标签 (one-hot)                        │   │
│  │  只学习"正确答案"                                   │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  知识蒸馏:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  学生模型 ← 软标签 (教师输出) + 硬标签              │   │
│  │  学习"正确答案" + "类间关系"                        │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  蒸馏的优势:                                                │
│  - 软标签提供更丰富的监督信号                              │
│  - 学生可以学到教师的"泛化能力"                            │
│  - 在小数据集上效果更明显                                  │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 直接训练学生模型 (不使用蒸馏)
# ============================================================
print("=" * 60)
print("直接训练学生模型 (无蒸馏)")
print("=" * 60)

# 创建新的学生模型
student_direct = StudentModel()
optimizer_direct = torch.optim.Adam(student_direct.parameters(), lr=1e-3)

print("\n训练中...")
student_direct.train()
direct_losses = []

for epoch in range(10):
    total_loss = 0
    for x_batch, y_batch in train_data:
        optimizer_direct.zero_grad()
        output = student_direct(x_batch)
        # 只使用硬标签损失
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer_direct.step()
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_data)
    direct_losses.append(avg_loss)
    
    if (epoch + 1) % 2 == 0:
        print(f"  Epoch {epoch+1}/10: Loss = {avg_loss:.4f}")

print("\n直接训练完成!")

In [ ]:
# ============================================================
# 比较蒸馏学生 vs 直接训练学生
# ============================================================
print("=" * 60)
print("蒸馏 vs 直接训练对比")
print("=" * 60)

# 测试数据
test_data = torch.randn(100, 784)

# 设置为评估模式
teacher.eval()
student.eval()
student_direct.eval()

with torch.no_grad():
    teacher_out = teacher(test_data)
    distilled_out = student(test_data)
    direct_out = student_direct(test_data)

# 计算与教师的输出差异
distilled_diff = (teacher_out - distilled_out).abs().mean().item()
direct_diff = (teacher_out - direct_out).abs().mean().item()

print(f"\n与教师模型输出的平均差异:")
print(f"  蒸馏学生:     {distilled_diff:.4f}")
print(f"  直接训练学生: {direct_diff:.4f}")

# 预测一致性
teacher_pred = teacher_out.argmax(dim=1)
distilled_pred = distilled_out.argmax(dim=1)
direct_pred = direct_out.argmax(dim=1)

distilled_match = (teacher_pred == distilled_pred).float().mean().item()
direct_match = (teacher_pred == direct_pred).float().mean().item()

print(f"\n与教师预测一致率:")
print(f"  蒸馏学生:     {distilled_match * 100:.1f}%")
print(f"  直接训练学生: {direct_match * 100:.1f}%")

print(f"\n结论:")
if distilled_match > direct_match:
    print(f"  蒸馏学生更接近教师模型的行为!")
else:
    print(f"  在此随机数据上差异不明显，真实数据集上蒸馏通常更优")

## 7. 超参数调优

**核心参数**: 温度 T 和软标签权重 α 对蒸馏效果影响很大

```
┌─────────────────────────────────────────────────────────────┐
│                   超参数选择指南                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  温度 T:                                                    │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  T 太小 (1-2): 软标签接近 one-hot，暗知识不足       │   │
│  │  T 太大 (>20): 分布过于平滑，主要类别信息丢失       │   │
│  │  推荐范围: T = 4 ~ 10                               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  软标签权重 α:                                              │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  α 太小 (<0.3): 主要依赖硬标签，蒸馏效果弱          │   │
│  │  α 太大 (>0.9): 忽略硬标签，可能偏离正确分类        │   │
│  │  推荐范围: α = 0.5 ~ 0.8                            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 温度参数对蒸馏效果的影响
# ============================================================
print("=" * 60)
print("温度参数对蒸馏效果的影响")
print("=" * 60)

temperatures = [1.0, 2.0, 4.0, 8.0, 16.0]
temp_results = []

print("\n测试不同温度...")
for T in temperatures:
    # 创建新的学生模型
    student_temp = StudentModel()
    optimizer_temp = torch.optim.Adam(student_temp.parameters(), lr=1e-3)
    
    # 快速训练 5 个 epoch
    student_temp.train()
    for epoch in range(5):
        for x_batch, y_batch in train_data:
            with torch.no_grad():
                teacher_logits_batch = teacher(x_batch)
            
            student_logits_batch = student_temp(x_batch)
            loss, _ = distillation_loss(
                student_logits_batch, teacher_logits_batch, y_batch,
                temperature=T, alpha=0.7
            )
            
            optimizer_temp.zero_grad()
            loss.backward()
            optimizer_temp.step()
    
    # 评估与教师的一致性
    student_temp.eval()
    with torch.no_grad():
        out = student_temp(test_data)
    match = (teacher_pred == out.argmax(dim=1)).float().mean().item()
    temp_results.append(match)
    print(f"  T={T}: 与教师一致率 = {match*100:.1f}%")

# 可视化
plt.figure(figsize=(10, 5))
plt.plot(temperatures, [r * 100 for r in temp_results], 'bo-', linewidth=2, markersize=10)
plt.xlabel('温度 T', fontsize=12)
plt.ylabel('与教师预测一致率 (%)', fontsize=12)
plt.title('温度参数对蒸馏效果的影响', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(temperatures)
plt.show()

print("\n观察: 通常 T=4~8 效果较好，过高或过低都会影响蒸馏质量")

## 8. 总结

### 蒸馏类型对比

| 类型 | 匹配内容 | 损失函数 | 优点 | 缺点 |
|:-----|:---------|:---------|:-----|:-----|
| 响应蒸馏 | 输出分布 | KL 散度 | 简单通用 | 信息量有限 |
| 特征蒸馏 | 中间层特征 | MSE | 传递更多知识 | 需要对齐维度 |
| 注意力迁移 | 空间注意力 | MSE | 适合 CNN | 仅限视觉模型 |
| 关系蒸馏 | 样本间关系 | MSE | 不依赖维度 | 计算复杂度高 |

### 超参数选择指南

| 参数 | 推荐范围 | 说明 |
|:-----|:---------|:-----|
| 温度 T | 4 ~ 10 | 控制软标签平滑度 |
| 软标签权重 α | 0.5 ~ 0.8 | 平衡软硬标签 |
| 学习率 | 1e-4 ~ 1e-3 | 通常比从头训练小 |
| 训练轮数 | 原始的 10-30% | 蒸馏收敛较快 |

### 最佳实践

```
蒸馏检查清单:
✓ 确保教师模型已经充分训练
✓ 选择合适的温度参数 (通常 4-10)
✓ 平衡软标签和硬标签损失
✓ 学生模型架构要合理 (不能太小)
✓ 在验证集上监控蒸馏效果

常见陷阱:
✗ 教师模型本身精度不高
✗ 温度参数设置不当
✗ 学生模型容量过小，无法学习
✗ 只用软标签，忽略硬标签
✗ 训练数据与教师训练数据分布不一致
```

### 应用场景选择

```
┌─────────────────────────────────────────────────────────────┐
│                   蒸馏方法选择指南                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  简单分类任务:                                              │
│  └── 响应蒸馏 (最简单，效果好)                             │
│                                                             │
│  复杂视觉任务 (CNN):                                        │
│  └── 响应蒸馏 + 注意力迁移                                 │
│                                                             │
│  NLP 任务 (Transformer):                                    │
│  └── 响应蒸馏 + 特征蒸馏                                   │
│                                                             │
│  小数据集:                                                  │
│  └── 关系蒸馏 (更鲁棒)                                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 下一步学习

- **04_Export_tutorial.ipynb**: 模型导出与部署
- **05_Advanced_Optimization_tutorial.ipynb**: 高级优化技术 (量化+剪枝+蒸馏组合)